In [21]:
import pandas as pd

papers = pd.read_csv("../outputs/final/paper_nodes_2022_2025.csv")

papers['raw_id'] = papers['node_id'].apply(lambda x: x.split('_')[-1])

raw_to_global = dict(zip(papers['raw_id'], papers['node_id']))
valid_raw_ids = set(raw_to_global.keys())

In [24]:
import pandas as pd
import ast

# Load metadata and papers
meta = pd.read_csv("../outputs/final/openalex_metadata_2022_2025.csv")
papers = pd.read_csv("../outputs/final/paper_nodes_2022_2025.csv")
knowledge=pd.read_csv("../outputs/final/knowledge_edges_2022_2025.csv")

valid_ids = set(papers['node_id'])

citation_edges = []

for _, row in meta.iterrows():
    source = row['global_paper_id']

    if pd.isna(row['referenced_works']):
        continue

    try:
        refs = ast.literal_eval(row['referenced_works'])
    except:
        continue

    for ref in refs:
        citation_edges.append({
            "source": source,
            "target": ref,     # keep raw OpenAlex ID
            "predicate": "cites",
            "year": row['year']
        })

citation_edges = pd.DataFrame(citation_edges)

print("Citation edges created:", len(citation_edges))
print("Papers with citation edges:", citation_edges['source'].nunique())

Citation edges created: 320724
Papers with citation edges: 2235


In [26]:
papers_with_knowledge = set(knowledge['source'])
papers_with_citation  = set(citation_edges['source'])

papers_with_any = papers_with_knowledge | papers_with_citation

print("Still isolated:", 2331 - len(papers_with_any))

Still isolated: 58


In [27]:
citation_edges.to_csv("../outputs/other/citation_edges_2022_2025.csv", index=False)
print("Saved citation_edges_2022_2025.csv")

Saved citation_edges_2022_2025.csv
